In [ ]:
#| default_exp skill

## Installing the skill

Install the notebook workflow instructions and their reference notes for agents.

The repository also ships the instructions that teach an agent how to use these tools. This notebook installs the bundled `SKILL.md` and references into local Codex or Claude skills directories, and can register the nbskill MCP server in Cursor's `mcp.json` so the notebook-first workflow can travel with the package.

The skill files are packaging for agent behavior, not another implementation path. They point agents back to the same notebook-aware tools in this repository, so users get consistent reads, writes, execution, and review whether they invoke nbskill through Python, the CLI, or MCP.

```python
build_skill_from_readme("README.md", "nbskill/SKILL.md")
install_nbskill(dest="~/.codex/skills/jupyter-notebooks")
```

### Production contract

Skill installation is CLI-first. It must build `SKILL.md` from the documented README section, install only into the requested Codex or Claude skills directory, write Cursor MCP configuration only when Cursor is requested, leave MCP processes running, print restart guidance when needed, and install hooks only when explicitly requested.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from nbskill.skill import build_skill_from_readme as _example_build_skill_from_readme
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
root = demo_path("06_skill_example")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text("before\n<!-- nbskill-skill:start -->\n# Skill Body\nUse read_nb first.\n<!-- nbskill-skill:end -->\nafter\n", encoding="utf-8")
    _example_build_skill_from_readme(str(readme), str(out))
    print("built:", out.name)
    print(out.read_text(encoding="utf-8").splitlines()[-2])
finally:
    remove_demo_path(root)

Built nbs/data/06_skill_example/SKILL.md from nbs/data/06_skill_example/README.md
built: SKILL.md
# Skill Body


In [ ]:
#| export
import json, subprocess
from importlib.resources import files
from pathlib import Path


from nbskill.foundation import cli_return, install_nbdev_pre_commit_hooks

In [ ]:
#| export
_SKILL_START = "<!-- nbskill-skill:start -->"
_SKILL_END = "<!-- nbskill-skill:end -->"
_SKILL_FRONTMATTER = """---
name: jupyter-notebooks
description: Work notebook-first in nbdev projects with nbskill MCP tools for reading, writing, updating, and executing notebooks without raw JSON.
---"""

In [ ]:
#| export
def _marked_readme_section(text, start=_SKILL_START, end=_SKILL_END):
    start_count = text.count(start)
    end_count = text.count(end)
    if start_count != 1 or end_count != 1:
        raise ValueError(f"README must contain exactly one {start!r} and one {end!r} marker")
    if text.index(start) > text.index(end):
        raise ValueError(f"README marker {start!r} must appear before {end!r}")
    return text.split(start, 1)[1].split(end, 1)[0].strip()

In [ ]:
#| export
def _skill_frontmatter(out_path):
    path = Path(out_path)
    if path.exists():
        text = path.read_text(encoding="utf-8")
        if text.startswith("---\n"):
            parts = text.split("\n---", 1)
            if len(parts) == 2: return f"---{parts[0][3:]}\n---"
    return _SKILL_FRONTMATTER

In [ ]:
#| export
def build_skill_from_readme(
    readme_path: str = "README.md",  # README containing the marked skill section
    out_path: str = "nbskill/SKILL.md",  # Skill file to write
):
    "Build SKILL.md from the marked section of README.md."
    out = Path(out_path)
    body = _marked_readme_section(Path(readme_path).read_text(encoding="utf-8"))
    text = f"{_skill_frontmatter(out)}\n\n{body}\n"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding="utf-8")
    print(f"Built {out} from {readme_path}")
    return cli_return(out)

### Installing agent instructions

The installer resolves the target skills directory, copies the packaged `SKILL.md`, and includes reference files. That keeps the project documentation and the agent workflow in sync with the package version.

In [ ]:
#| export
def _nbskill_mcp_pids():
    try:
        from nbskill.cli import _find_nbskill_mcp_processes
        return _find_nbskill_mcp_processes(all_projects=True)
    except Exception:
        pass
    proc = subprocess.run(["ps", "-eo", "pid=,ppid=,command="], text=True, capture_output=True)
    if proc.returncode != 0: return []
    pids = []
    for line in proc.stdout.splitlines():
        parts = line.strip().split(None, 2)
        if len(parts) != 3: continue
        try: pid, ppid = int(parts[0]), int(parts[1])
        except ValueError: continue
        command = parts[2]
        if "nbskill_mcp" not in command and "nbskill.mcp" not in command: continue
        if any(name in command for name in ("nbskill_mcp_start", "nbskill_mcp_stop", "nbskill_mcp_restart")): continue
        pids.append({"pid": pid, "ppid": ppid, "command": command.strip()})
    return pids

In [ ]:
#| export
def nbskill_mcp_restart_notice(timeout=5.0, force=True, dry_run=False):
    "Restart running nbskill MCP server processes and return a user-facing result."
    try:
        from nbskill.cli import _stop_nbskill_mcp_processes
        result = _stop_nbskill_mcp_processes(all_projects=True, timeout=timeout, force=force, dry_run=dry_run)
    except Exception as exc:
        running = _nbskill_mcp_pids()
        return {
            "running": bool(running),
            "restarted": False,
            "processes": running,
            "message": f"Could not restart nbskill MCP processes automatically: {exc}",
        }
    matched = result.get("matched", [])
    alive = result.get("alive", [])
    if not matched:
        return {"running": False, "restarted": False, "processes": [], "message": "No running nbskill MCP server process detected.", "result": result}
    if result.get("dry_run"):
        message = f"Would restart {len(matched)} nbskill MCP server process(es)."
    elif alive:
        message = f"Stopped {len(result.get('terminated', []))} nbskill MCP process target(s), but {len(alive)} process(es) are still alive."
    else:
        message = f"Restarted {len(matched)} nbskill MCP server process(es); the MCP client will start fresh nbskill_mcp processes on reconnect."
    return {"running": True, "restarted": not bool(alive) and not result.get("dry_run"), "processes": matched, "message": message, "result": result}

In [ ]:
#| export
def _nbskill_source_project():
    """Return the current checkout when running from the nbskill source tree."""
    for root in [Path.cwd(), *Path.cwd().parents]:
        pyproject = root / "pyproject.toml"
        if not pyproject.exists(): continue
        try: text = pyproject.read_text(encoding="utf-8")
        except OSError: continue
        if 'name = "nbskill"' in text and (root / "nbskill" / "mcp.py").exists():
            return root.resolve()
    return None

In [ ]:
#| export
def _cursor_mcp_path(workspace=None):
    root = Path(workspace).expanduser() if workspace else Path.home()
    return root / ".cursor" / "mcp.json"

In [ ]:
#| export
def _cursor_nbskill_server_config():
    return {"command": "nbskill_mcp"}

In [ ]:
#| export
def _install_cursor_mcp(workspace=None, overwrite=True):
    """Install nbskill's MCP server into Cursor's mcp.json."""
    path = _cursor_mcp_path(workspace)
    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict): raise ValueError(f"Cursor MCP config must be a JSON object: {path}")
    else: data = {}
    servers = data.setdefault("mcpServers", {})
    if not isinstance(servers, dict): raise ValueError(f"Cursor MCP config mcpServers must be an object: {path}")
    if "nbskill" in servers and not overwrite: raise FileExistsError(path)
    servers["nbskill"] = _cursor_nbskill_server_config()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {
        "installed": True,
        "path": path,
        "server": "nbskill",
        "workspace": str(Path(workspace).expanduser()) if workspace else None,
    }

In [ ]:
#| export
def install_nbskill(
    target: str = "codex",  # codex, claude, cursor, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Overwrite an existing SKILL.md or Cursor server entry
    install_hooks: bool = False,  # Install nbdev-clean/nbdev-test pre-commit hooks in the current repo
    restart_mcp: bool = True,  # Restart running nbskill MCP servers after updating install/config
    cursor_workspace: str | None = None,  # Cursor workspace for .cursor/mcp.json; omit for global ~/.cursor/mcp.json
):
    "Install nbskill agent instructions or Cursor MCP configuration."
    target = target.lower()
    cursor_mcp = {"installed": False, "reason": "target-not-cursor"}
    if target == "cursor": roots = []
    elif skills_dir: roots = [Path(skills_dir).expanduser()]
    elif target == "codex": roots = [Path.home() / ".codex" / "skills"]
    elif target in {"claude", "claude-code", "claude_code"}: roots = [Path.home() / ".claude" / "skills"]
    elif target == "both": roots = [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    else: raise ValueError("target must be codex, claude, cursor, both, or use skills_dir")
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed = []
    for root in roots:
        dst_dir = root / skill_name
        dst = dst_dir / "SKILL.md"
        if dst.exists() and not overwrite: raise FileExistsError(dst)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst.write_text(skill_text, encoding="utf-8")
        if references.is_dir():
            ref_dir = dst_dir / "references"
            ref_dir.mkdir(exist_ok=True)
            for ref in references.iterdir():
                if ref.is_file():
                    (ref_dir / ref.name).write_text(ref.read_text(encoding="utf-8"), encoding="utf-8")
        installed.append(dst)
    if target == "cursor": cursor_mcp = _install_cursor_mcp(workspace=cursor_workspace, overwrite=overwrite)
    hooks = install_nbdev_pre_commit_hooks(Path.cwd()) if install_hooks else {"installed": False, "reason": "disabled"}
    mcp_notice = {"running": False, "message": "not checked"}
    if restart_mcp and not skills_dir and target in {"codex", "both", "cursor"}:
        mcp_notice = nbskill_mcp_restart_notice()
    for path in installed: print(f"Installed {path}")
    if cursor_mcp.get("installed"): print(f"Installed Cursor MCP config at {cursor_mcp['path']}")
    if hooks.get("installed"): print(f"Installed nbdev pre-commit hook at {hooks['hook']}")
    else: print(f"Skipped nbdev pre-commit hook: {hooks.get('reason')}")
    if mcp_notice.get("running"):
        print("nbskill MCP restarted:" if mcp_notice.get("restarted") else "nbskill MCP restart incomplete:")
        print(mcp_notice["message"])
        for proc in mcp_notice.get("processes", []):
            cwd = f" cwd={proc['cwd']}" if proc.get("cwd") else ""
            print(f"- pid={proc['pid']}{cwd} command={proc['command']}")
    return cli_return({"installed": installed, "cursor_mcp": cursor_mcp, "hooks": hooks, "mcp_restart": mcp_notice})

In [ ]:
#| eval: false
install_nbskill(target="cursor", cursor_workspace=".")

In [ ]:
install_root = demo_path("06_skill_install")
try:
    install_nbskill(skills_dir=str(install_root))
    skill_dir = install_root / "jupyter-notebooks"
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "mcp-tools.md").exists()
    assert (skill_dir / "references" / "cli-fallbacks.md").exists()
    assert (skill_dir / "references" / "conversion.md").exists()
    assert (skill_dir / "references" / "extended-tools.md").exists()
finally:
    remove_demo_path(install_root)


In [ ]:
cursor_root = demo_path("06_skill_cursor_install")
try:
    (cursor_root / ".cursor").mkdir(parents=True)
    existing = {"mcpServers": {"other": {"command": "python", "args": ["server.py"]}}}
    (cursor_root / ".cursor" / "mcp.json").write_text(json.dumps(existing), encoding="utf-8")
    install_nbskill(target="cursor", cursor_workspace=str(cursor_root), restart_mcp=False)
    config = json.loads((cursor_root / ".cursor" / "mcp.json").read_text(encoding="utf-8"))
    assert config["mcpServers"]["other"]["command"] == "python"
    assert config["mcpServers"]["nbskill"] == {"command": "nbskill_mcp"}
finally:
    remove_demo_path(cursor_root)

In [ ]:
root = demo_path("06_skill_build")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text(
        """before
<!-- nbskill-skill:start -->
# Skill Body
Use notebooks.
<!-- nbskill-skill:end -->
after
""",
        encoding="utf-8",
    )
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: jupyter-notebooks")
    assert "# Skill Body" in text
    assert "before" not in text and "after" not in text
    out.write_text("---\nname: custom-skill\ndescription: Keep me.\n---\nold\n", encoding="utf-8")
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: custom-skill")
    assert "# Skill Body" in text and "old" not in text
finally:
    remove_demo_path(root)


In [ ]:
root = demo_path("06_skill_duplicate_markers")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    marked = "<!-- nbskill-skill:start -->\n# Body\n<!-- nbskill-skill:end -->\n"
    readme.write_text(marked + marked, encoding="utf-8")
    try: build_skill_from_readme(str(readme), str(out))
    except ValueError as exc: assert "exactly one" in str(exc)
    else: raise AssertionError("duplicate skill markers should fail")
finally:
    remove_demo_path(root)

In [ ]:
import tomllib
from pathlib import Path


In [ ]:
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
scripts = tomllib.loads((project_root / "pyproject.toml").read_text(encoding="utf-8"))["project"]["scripts"]
assert "batch_edit_nb" in scripts
assert "nbskill_status" in scripts
assert "install_nbskill" in scripts


In [ ]:
for removed in ("nbskill-mcp", "private-symbol-report", "symbol-graph", "update-cell", "show-doc"):
    for path in [project_root / "README.md", project_root / "nbskill/SKILL.md", *(project_root / "nbskill/references").glob("*.md")]:
        if path.exists(): assert removed not in path.read_text(encoding="utf-8")


In [ ]:
hook_root = demo_path("06_skill_hooks")
try:
    hook_root.mkdir()
    (hook_root / "nbs").mkdir()
    subprocess.run(["git", "init"], cwd=hook_root, check=True, capture_output=True)
    hook_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    pre_commit = hook_root / ".git" / "hooks" / "pre-commit"
    hook_text = pre_commit.read_text(encoding="utf-8")
    assert hook_result["installed"]
    assert "nbdev-clean" in hook_text
    assert "nbdev-test" in hook_text
    assert (hook_root / ".git" / "info" / "nbskill-hooks-installed").exists()

    pre_commit.unlink()
    removed_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    assert not removed_result["installed"]
    assert removed_result["reason"] == "hooks-removed-by-user"
    assert not pre_commit.exists()
    assert (hook_root / ".git" / "info" / "nbskill-hooks-disabled").exists()
finally:
    remove_demo_path(hook_root)


In [ ]:
install_root = demo_path("06_skill_custom_install")
try:
    install_nbskill(
        target="custom", skills_dir=str(install_root), install_hooks=False, restart_mcp=False,
    )
    assert (install_root / "jupyter-notebooks" / "SKILL.md").exists()
    notice = nbskill_mcp_restart_notice(dry_run=True)
    assert "running" in notice and "message" in notice
finally:
    remove_demo_path(install_root)